# Phase 2: Grounded-SAM masks + hint-based colorization (Colab)

Consumes the 5 hand-authored plans in `examples/plans/phase2/`:
1. **Grounded-SAM**: each region's `grounding_phrase` → Grounding DINO box → SAM mask. Uses HuggingFace `transformers` ports (no CUDA-extension compilation).
2. **Naive render + adherence** (sanity check, runs immediately).
3. **Control Color**: hint image (strokes from masks) → diffusion colorizer. Scaffolded; needs live iteration like L-CAD did.

Grounding runs on the **grayscale** image — the honest test-time input.

> Runtime → GPU before running.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/tomqi6195/chroma-reasoner.git
%cd /content
!pip install -q -e chroma-reasoner
!pip install -q pycocotools
# restart-free editable install: make the package importable now
import sys; sys.path.insert(0, '/content/chroma-reasoner/src')

## 1. Fetch the 5 images + make grayscale

In [ ]:
import glob, json, os, urllib.request
import cv2

PLANS = sorted(glob.glob('chroma-reasoner/examples/plans/phase2/*.json'))
IMAGE_IDS = [json.load(open(p))['image_id'] for p in PLANS]
print(IMAGE_IDS)

os.makedirs('images', exist_ok=True); os.makedirs('gray', exist_ok=True)
for iid in IMAGE_IDS:
    dest = f'images/{iid}.jpg'
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f'http://images.cocodataset.org/val2017/{iid}.jpg', dest)
    img = cv2.imread(dest)
    lab_l = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[:, :, 0]
    cv2.imwrite(f'gray/{iid}.png', lab_l)
print('done')

## 2. Grounded-SAM: grounding_phrase → box → mask

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import (AutoModelForZeroShotObjectDetection, AutoProcessor,
                          SamModel, SamProcessor)

device = 'cuda'
dino_id = 'IDEA-Research/grounding-dino-base'
sam_id = 'facebook/sam-vit-huge'   # drop to sam-vit-base if VRAM-tight
dino_proc = AutoProcessor.from_pretrained(dino_id)
dino = AutoModelForZeroShotObjectDetection.from_pretrained(dino_id).to(device).eval()
sam_proc = SamProcessor.from_pretrained(sam_id)
sam = SamModel.from_pretrained(sam_id).to(device).eval()

@torch.no_grad()
def phrase_to_mask(image_pil, phrase):
    # Grounding DINO wants lowercase queries terminated with '.'
    text = phrase.lower().rstrip('.') + '.'
    inputs = dino_proc(images=image_pil, text=text, return_tensors='pt').to(device)
    out = dino(**inputs)
    res = dino_proc.post_process_grounded_object_detection(
        out, inputs.input_ids, threshold=0.25, text_threshold=0.2,
        target_sizes=[image_pil.size[::-1]])[0]
    if len(res['boxes']) == 0:
        return None, 0.0
    best = res['scores'].argmax()
    box = res['boxes'][best].tolist()
    s_in = sam_proc(image_pil, input_boxes=[[box]], return_tensors='pt').to(device)
    s_out = sam(**s_in)
    masks = sam_proc.image_processor.post_process_masks(
        s_out.pred_masks.cpu(), s_in['original_sizes'].cpu(), s_in['reshaped_input_sizes'].cpu())[0][0]
    scores = s_out.iou_scores.cpu()[0, 0]
    return masks[scores.argmax()].numpy().astype(bool), float(res['scores'][best])

In [ ]:
from chroma_reasoner.plan import load_plan
from chroma_reasoner.plan.masks import region_key, save_mask

MASKS_ROOT = 'masks'
for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    # ground on the grayscale image, replicated to 3 channels
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    pil = Image.fromarray(np.stack([gray] * 3, axis=-1))
    for region in plan['regions']:
        mask, score = phrase_to_mask(pil, region['grounding_phrase'])
        if mask is None:
            print(f"!! {iid}/{region_key(region)}: NO DETECTION - simplify the phrase")
            continue
        save_mask(mask, MASKS_ROOT, iid, region)
        print(f"{iid}/{region_key(region)}: dino score {score:.2f}, mask {mask.sum()} px")

In [ ]:
# Visual check: overlay every mask on its image. Wrong masks => fix phrases
# in the plan JSON and re-run the cell above.
import matplotlib.pyplot as plt

for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    img = cv2.cvtColor(cv2.imread(f'images/{iid}.jpg'), cv2.COLOR_BGR2RGB)
    regions = plan['regions']
    fig, axes = plt.subplots(1, len(regions) + 1, figsize=(4 * (len(regions) + 1), 4))
    axes[0].imshow(img); axes[0].set_title(iid); axes[0].axis('off')
    for ax, region in zip(axes[1:], regions):
        key = region_key(region)
        path = f'{MASKS_ROOT}/{iid}/{key}.png'
        ax.axis('off'); ax.set_title(key)
        if os.path.exists(path):
            m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) > 127
            over = img.copy(); over[m] = (0.4 * over[m] + 0.6 * np.array([255, 0, 0])).astype(np.uint8)
            ax.imshow(over)
    plt.show()

## 3. Naive render + adherence (instant end-to-end check)

In [ ]:
from chroma_reasoner.plan.adherence import evaluate_adherence
from chroma_reasoner.plan.hints import render_naive
from chroma_reasoner.plan.masks import load_masks

os.makedirs('results/naive', exist_ok=True)
for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    try:
        masks = load_masks(MASKS_ROOT, iid, plan, shape=gray.shape)
    except FileNotFoundError as e:
        print(f'{iid}: {e}'); continue
    rgb = render_naive(gray, masks, plan)
    cv2.imwrite(f'results/naive/{iid}.png', cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    report = evaluate_adherence(rgb, masks, plan)
    print(iid, '| mean dE', report['mean_delta_e'], '| pass', report['n_pass'], '/', report['n_regions'])
    plt.figure(figsize=(6, 6)); plt.imshow(rgb); plt.axis('off'); plt.show()

## 4. Control Color (diffusion colorizer)

Interface (from reading `test.py`): `process(...)` takes `input_image` (grayscale, 3-channel), `hint_image` (same image with colour strokes painted on), and a text `prompt`; it recovers the hinted pixels by comparing the two. Our `make_hint_image` builds exactly that from masks + plan.

Checkpoints (`main_model.ckpt`, `content-guided_deformable_vae.ckpt`) come from the Google Drive linked in the repo README → `pretrained_models/`. **Expect the Drive quota problem again** — use the same Make-a-copy workaround as L-CAD if gdown fails.

Like L-CAD, this repo predates today's Colab; expect to iterate on imports/versions on first run (`transformers<5` will likely be needed again for its CLIP).

In [ ]:
!git clone https://github.com/ZhexinLiang/Control-Color.git
# TODO(first run): download the two ckpts from the README's Google Drive link
# into Control-Color/pretrained_models/ (gdown or Drive Make-a-copy + mount).
!ls Control-Color

In [ ]:
# Build hint images for every plan (works now, independent of Control Color).
from chroma_reasoner.plan.hints import make_hint_image

os.makedirs('hints', exist_ok=True)
for plan_path in PLANS:
    plan = load_plan(plan_path)
    iid = plan['image_id']
    gray = cv2.imread(f'gray/{iid}.png', cv2.IMREAD_GRAYSCALE)
    gray_rgb = np.stack([gray] * 3, axis=-1)
    masks = load_masks(MASKS_ROOT, iid, plan, shape=gray.shape)
    hint = make_hint_image(gray_rgb, masks, plan, erosion=0.15)
    cv2.imwrite(f'hints/{iid}_input.png', cv2.cvtColor(gray_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f'hints/{iid}_hint.png', cv2.cvtColor(hint, cv2.COLOR_RGB2BGR))
    plt.figure(figsize=(6, 6)); plt.imshow(hint); plt.axis('off'); plt.title(f'{iid} hint'); plt.show()

# TODO(first run): adapt Control-Color/test.py — extract its model setup and
# call process(change_according_to_strokes=True, iterative_editing=False,
#              input_image=<hints/{iid}_input.png>, hint_image=<hints/{iid}_hint.png>, ...)
# then save outputs to results/ctrlcolor/{iid}.png and run evaluate_adherence on them.

## 5. Export masks + outputs

In [ ]:
!zip -rq phase2_outputs.zip masks results hints
from google.colab import files
files.download('phase2_outputs.zip')
# Locally: extract into the repo root, then e.g.
#   python scripts/render_naive.py --plan examples/plans/phase2/000000010092.json \
#       --gray data/coco/gray/000000010092.png --masks masks --out results/phase2/naive